# 02 — Statistika Deskriptif Berkelompok
**PSAJ Statistika | Bab 4.1 & 4.2 Laporan**

Notebook ini menghasilkan:
- Nilai Min, Max, Range
- Aturan Sturges (Jumlah & Panjang Kelas)
- Tabel Distribusi Frekuensi Kelompok
- **Mean, Median, Modus** berkelompok (step-by-step)

---
**Input:** `outputs/merged_2025.csv`

In [ ]:
import pandas as pd
import numpy as np
import math

# Load data bersih hasil notebook 01
df = pd.read_csv('outputs/merged_2025.csv')
X = df['SMA_Plus_Pct'].values   # Variabel X: Persentase Pendidikan SMA+
n = len(X)

print(f'Jumlah data (n): {n}')
print(df[['Provinsi', 'SMA_Plus_Pct', 'TPT_Pct']].to_string())

## 4.1 — Eksplorasi Data Awal

In [ ]:
# =============================================
# NILAI MINIMUM, MAKSIMUM, DAN JANGKAUAN (RANGE)
# =============================================
x_min  = X.min()
x_max  = X.max()
range_ = x_max - x_min

print('=== EKSPLORASI DATA AWAL ===')
print(f'Provinsi dengan SMA+ terendah : {df.loc[X.argmin(), "Provinsi"]} ({x_min:.2f}%)')
print(f'Provinsi dengan SMA+ tertinggi: {df.loc[X.argmax(), "Provinsi"]} ({x_max:.2f}%)')
print(f'Nilai Minimum (Xmin)           : {x_min:.2f}')
print(f'Nilai Maksimum (Xmax)          : {x_max:.2f}')
print(f'Jangkauan/Range (R = Xmax-Xmin): {range_:.2f}')

## 4.2 — Tabel Distribusi Frekuensi & Pemusatan Data

In [ ]:
# =============================================
# ATURAN STURGES
# k = 1 + 3.3 * log10(n)
# =============================================
k_float  = 1 + 3.3 * math.log10(n)
k        = math.ceil(k_float)          # Jumlah kelas (dibulatkan ke atas)
p_float  = range_ / k
p        = math.ceil(p_float)          # Panjang kelas (dibulatkan ke atas)

print('=== ATURAN STURGES ===')
print(f'k = 1 + 3.3 × log₁₀({n})')
print(f'k = 1 + 3.3 × {math.log10(n):.4f}')
print(f'k = {k_float:.4f}  →  dibulatkan menjadi  k = {k} kelas')
print(f'\nPanjang Kelas (p = R/k):')
print(f'p = {range_:.2f} / {k} = {p_float:.4f}  →  dibulatkan menjadi  p = {p}')

In [ ]:
# =============================================
# TABEL DISTRIBUSI FREKUENSI KELOMPOK
# =============================================

# Batas bawah kelas pertama = x_min (atau dibulatkan ke bawah)
batas_bawah_awal = math.floor(x_min)

# Buat bins (tepi kelas)
bins = [batas_bawah_awal + i * p for i in range(k + 1)]
labels = [f'{bins[i]:.1f} – {bins[i+1]:.1f}' for i in range(k)]

# Hitung frekuensi
freq, _ = np.histogram(X, bins=bins)

# Tepi bawah (Tb) dan tepi atas (Ta)
tepi_bawah = [bins[i] - 0.5 for i in range(k)]
tepi_atas  = [bins[i+1] - 0.5 for i in range(k)]

# Titik tengah (xi)
titik_tengah = [(tepi_bawah[i] + tepi_atas[i]) / 2 for i in range(k)]

# Frekuensi kumulatif kurang dari dan lebih dari
fk_kurang = np.cumsum(freq)
fk_lebih  = n - np.cumsum(freq) + freq  # frekuensi kumulatif lebih dari

# f × xi
fi_xi = freq * np.array(titik_tengah)

# Susun tabel
tabel = pd.DataFrame({
    'Interval Kelas': labels,
    'Tepi Bawah (Tb)': tepi_bawah,
    'Tepi Atas (Ta)' : tepi_atas,
    'Titik Tengah (xi)': [round(t, 2) for t in titik_tengah],
    'Frekuensi (fi)' : freq,
    'fi × xi'        : [round(v, 2) for v in fi_xi],
    'fk < (kurang dari)' : fk_kurang,
    'fk > (lebih dari)'  : fk_lebih,
})

# Baris total
total_row = pd.DataFrame([{
    'Interval Kelas': 'TOTAL',
    'Tepi Bawah (Tb)': '',
    'Tepi Atas (Ta)': '',
    'Titik Tengah (xi)': '',
    'Frekuensi (fi)': freq.sum(),
    'fi × xi': round(fi_xi.sum(), 2),
    'fk < (kurang dari)': '',
    'fk > (lebih dari)': '',
}])

tabel_full = pd.concat([tabel, total_row], ignore_index=True)
print('=== TABEL DISTRIBUSI FREKUENSI KELOMPOK ===')
print(tabel_full.to_string(index=False))

# Simpan ke CSV
tabel.to_csv('outputs/tables/tabel_frekuensi.csv', index=False)
print('\nTabel tersimpan ke outputs/tables/tabel_frekuensi.csv')

In [ ]:
# =============================================
# MEAN BERKELOMPOK
# x̄ = Σ(fi × xi) / Σfi
# =============================================
sum_fi    = freq.sum()
sum_fi_xi = fi_xi.sum()

mean_kelompok = sum_fi_xi / sum_fi

print('=== MEAN BERKELOMPOK ===')
print(f'Σfi        = {sum_fi}')
print(f'Σ(fi × xi) = {sum_fi_xi:.2f}')
print(f'x̄ = Σ(fi × xi) / Σfi')
print(f'x̄ = {sum_fi_xi:.2f} / {sum_fi}')
print(f'x̄ = {mean_kelompok:.4f} ≈ {mean_kelompok:.2f}%')

In [ ]:
# =============================================
# MEDIAN BERKELOMPOK
# Me = Tb + ((n/2 - Fk) / f_me) × p
# =============================================

# Cari kelas median: kelas di mana fk_kurang pertama kali >= n/2
half_n     = n / 2
idx_median = np.searchsorted(fk_kurang, half_n)  # index kelas median

Tb_me  = tepi_bawah[idx_median]
Fk_me  = fk_kurang[idx_median - 1] if idx_median > 0 else 0  # fk sebelum kelas median
f_me   = freq[idx_median]

median_kelompok = Tb_me + ((half_n - Fk_me) / f_me) * p

print('=== MEDIAN BERKELOMPOK ===')
print(f'n/2                                        = {half_n}')
print(f'Kelas Median                               = Kelas {idx_median + 1} ({labels[idx_median]})')
print(f'Tepi Bawah Kelas Median (Tb)               = {Tb_me}')
print(f'Frekuensi Kumulatif sebelum kelas (Fk)     = {Fk_me}')
print(f'Frekuensi Kelas Median (f_me)              = {f_me}')
print(f'Panjang Kelas (p)                          = {p}')
print(f'\nMe = Tb + ((n/2 - Fk) / f_me) × p')
print(f'Me = {Tb_me} + (({half_n} - {Fk_me}) / {f_me}) × {p}')
print(f'Me = {Tb_me} + ({half_n - Fk_me} / {f_me}) × {p}')
print(f'Me = {Tb_me} + {(half_n - Fk_me) / f_me:.4f} × {p}')
print(f'Me = {median_kelompok:.4f} ≈ {median_kelompok:.2f}%')

In [ ]:
# =============================================
# MODUS BERKELOMPOK
# Mo = Tb + (d1 / (d1 + d2)) × p
# =============================================

# Kelas modus = kelas dengan frekuensi tertinggi
idx_modus = np.argmax(freq)
Tb_mo     = tepi_bawah[idx_modus]
f_mo      = freq[idx_modus]
d1        = f_mo - (freq[idx_modus - 1] if idx_modus > 0 else 0)
d2        = f_mo - (freq[idx_modus + 1] if idx_modus < k - 1 else 0)

modus_kelompok = Tb_mo + (d1 / (d1 + d2)) * p

print('=== MODUS BERKELOMPOK ===')
print(f'Kelas Modus (frekuensi tertinggi)          = Kelas {idx_modus + 1} ({labels[idx_modus]}), fi = {f_mo}')
print(f'Tepi Bawah Kelas Modus (Tb)                = {Tb_mo}')
print(f'd1 = f_mo - f_(mo-1)                       = {f_mo} - {freq[idx_modus - 1] if idx_modus > 0 else 0} = {d1}')
print(f'd2 = f_mo - f_(mo+1)                       = {f_mo} - {freq[idx_modus + 1] if idx_modus < k - 1 else 0} = {d2}')
print(f'\nMo = Tb + (d1 / (d1 + d2)) × p')
print(f'Mo = {Tb_mo} + ({d1} / ({d1} + {d2})) × {p}')
print(f'Mo = {Tb_mo} + ({d1} / {d1+d2}) × {p}')
print(f'Mo = {Tb_mo} + {d1/(d1+d2):.4f} × {p}')
print(f'Mo = {modus_kelompok:.4f} ≈ {modus_kelompok:.2f}%')

In [ ]:
# =============================================
# RANGKUMAN HASIL (untuk Bab 4.2 laporan)
# =============================================
print('\n===== RANGKUMAN STATISTIKA DESKRIPTIF =====')
print(f'Nilai Minimum   : {x_min:.2f}%')
print(f'Nilai Maksimum  : {x_max:.2f}%')
print(f'Jangkauan (R)   : {range_:.2f}')
print(f'Jumlah Kelas (k): {k}')
print(f'Panjang Kelas(p): {p}')
print(f'Mean  (x̄)      : {mean_kelompok:.2f}%')
print(f'Median (Me)     : {median_kelompok:.2f}%')
print(f'Modus  (Mo)     : {modus_kelompok:.2f}%')

# Simpan rangkuman
rangkuman = pd.DataFrame([{
    'Xmin': x_min, 'Xmax': x_max, 'Range': range_,
    'k (kelas)': k, 'p (panjang kelas)': p,
    'Mean': round(mean_kelompok, 4),
    'Median': round(median_kelompok, 4),
    'Modus': round(modus_kelompok, 4)
}])
rangkuman.to_csv('outputs/tables/rangkuman_deskriptif.csv', index=False)
print('\nRangkuman tersimpan ke outputs/tables/rangkuman_deskriptif.csv')